# 🐦 Sentiment Analysis on Twitter Data
### Codect Technologies AI Internship Project
**Techniques Used:** NLP | NLTK | TextBlob | Naive Bayes | Visualization

---

## 📦 Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import re
import string
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.sentiment.vader import SentimentIntensityAnalyzer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Download NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt_tab', quiet=True)

matplotlib.rcParams['figure.facecolor'] = '#0f172a'
matplotlib.rcParams['axes.facecolor'] = '#1e293b'
matplotlib.rcParams['text.color'] = 'white'
matplotlib.rcParams['axes.labelcolor'] = 'white'
matplotlib.rcParams['xtick.color'] = 'white'
matplotlib.rcParams['ytick.color'] = 'white'

print('✅ All libraries imported successfully!')

## 🐦 Step 2: Create Twitter Dataset

In [ ]:
tweets_data = {
    'tweet': [
        # Positive tweets
        "I absolutely love this new phone! Best purchase ever! 😍",
        "Amazing weather today! Feeling so happy and grateful 🌞",
        "Just got promoted at work! Dreams do come true! 🎉",
        "This movie was absolutely fantastic! Highly recommend it!",
        "Great customer service! They solved my problem instantly 👍",
        "Python programming is so fun and powerful! Love coding!",
        "Had the most wonderful dinner with family tonight ❤️",
        "The new AI features are incredible! Technology is amazing!",
        "Just finished my workout! Feeling energized and strong 💪",
        "Beautiful sunset today! Nature never fails to amaze me 🌅",
        "My team won the championship! So proud of everyone! 🏆",
        "This book changed my life! Absolutely brilliant writing!",
        "Celebrating 5 years at my dream job today! Grateful! 🙏",
        "The concert was mind-blowing! Best night of my life!",
        "Finally passed my exams! Hard work really pays off! 📚",

        # Negative tweets
        "This product is terrible! Complete waste of money! 😡",
        "Worst customer service ever! Never buying from them again!",
        "I hate Mondays! Everything is going wrong today 😤",
        "The food at this restaurant was disgusting! Avoid it!",
        "My flight got cancelled again! This airline is horrible!",
        "Stuck in traffic for 3 hours! This city is a nightmare!",
        "The movie was boring and a complete waste of time! 😒",
        "Lost my job today. Feeling so hopeless and depressed.",
        "This app keeps crashing! Terrible user experience! 👎",
        "Failed my exam after weeks of studying. So disappointed.",
        "The internet has been down all day! So frustrated! 😠",
        "Terrible weather ruining my plans again! I hate this!",
        "Got scammed online. Never trusting any website again!",
        "The concert was awful! Poor sound quality throughout!",
        "Worst birthday ever! Nothing went as planned today.",

        # Neutral tweets
        "Just woke up and having my morning coffee. Monday again.",
        "The meeting has been rescheduled to tomorrow afternoon.",
        "Currently reading a book about machine learning concepts.",
        "The temperature today is around 25 degrees Celsius.",
        "Ordered pizza for dinner. Delivery takes 30 minutes.",
        "The new software update is available for download now.",
        "Going to the gym after work today as usual.",
        "The quarterly report will be released next week.",
        "Attended a webinar on data science this afternoon.",
        "The store closes at 9 PM on weekdays.",
        "Just submitted my assignment before the deadline.",
        "The bus arrives every 15 minutes on this route.",
        "Working from home today. Video call at 3 PM.",
        "The package was delivered to the front door.",
        "Finished reading the news this morning as usual."
    ],
    'sentiment': ['positive'] * 15 + ['negative'] * 15 + ['neutral'] * 15
}

df = pd.DataFrame(tweets_data)
print(f'✅ Dataset created with {len(df)} tweets!')
print(f'\n📊 Sentiment Distribution:')
print(df['sentiment'].value_counts())
df.head(10)

## 🧹 Step 3: Text Preprocessing

In [ ]:
def preprocess_tweet(tweet):
    """Clean and preprocess tweet text."""
    # Lowercase
    tweet = tweet.lower()
    # Remove URLs
    tweet = re.sub(r'http\S+|www\S+', '', tweet)
    # Remove mentions and hashtags
    tweet = re.sub(r'@\w+|#\w+', '', tweet)
    # Remove emojis and special characters
    tweet = tweet.encode('ascii', 'ignore').decode('ascii')
    # Remove punctuation
    tweet = tweet.translate(str.maketrans('', '', string.punctuation))
    # Remove extra whitespace
    tweet = ' '.join(tweet.split())
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    words = word_tokenize(tweet)
    words = [w for w in words if w not in stop_words and len(w) > 2]
    # Stemming
    stemmer = PorterStemmer()
    words = [stemmer.stem(w) for w in words]
    return ' '.join(words)

df['cleaned_tweet'] = df['tweet'].apply(preprocess_tweet)

print('✅ Preprocessing done!')
print('\nSample:')
print(f'Original : {df["tweet"][0]}')
print(f'Cleaned  : {df["cleaned_tweet"][0]}')

## 🤖 Step 4: VADER Sentiment Analysis

In [ ]:
sia = SentimentIntensityAnalyzer()

def get_vader_sentiment(tweet):
    scores = sia.polarity_scores(tweet)
    if scores['compound'] >= 0.05:
        return 'positive'
    elif scores['compound'] <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df['vader_sentiment'] = df['tweet'].apply(get_vader_sentiment)
df['vader_score'] = df['tweet'].apply(lambda x: sia.polarity_scores(x)['compound'])

# VADER accuracy
vader_acc = accuracy_score(df['sentiment'], df['vader_sentiment'])
print(f'✅ VADER Sentiment Analysis Complete!')
print(f'📊 VADER Accuracy: {vader_acc:.2%}')
print('\nSample Results:')
df[['tweet', 'sentiment', 'vader_sentiment', 'vader_score']].head(10)

## 🧠 Step 5: Naive Bayes Classifier

In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
X = tfidf.fit_transform(df['cleaned_tweet'])
y = df['sentiment']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Train Naive Bayes
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train, y_train)

# Predictions
y_pred = nb_model.predict(X_test)
nb_acc = accuracy_score(y_test, y_pred)

print(f'✅ Naive Bayes Model Trained!')
print(f'📊 Accuracy: {nb_acc:.2%}')
print(f'\n📋 Classification Report:')
print(classification_report(y_test, y_pred))

## 🔮 Step 6: Predict on New Tweets

In [ ]:
def predict_sentiment(tweet):
    """Predict sentiment of any new tweet."""
    cleaned = preprocess_tweet(tweet)
    vectorized = tfidf.transform([cleaned])
    prediction = nb_model.predict(vectorized)[0]
    vader = get_vader_sentiment(tweet)
    score = sia.polarity_scores(tweet)['compound']
    
    emoji = {'positive': '😊', 'negative': '😠', 'neutral': '😐'}[prediction]
    print(f'Tweet    : {tweet}')
    print(f'NB Model : {prediction.upper()} {emoji}')
    print(f'VADER    : {vader.upper()} (score: {score:.3f})')
    print('-' * 60)

# Test on new tweets
test_tweets = [
    "This AI internship is amazing! Learning so much every day!",
    "I can't believe how bad this service is! Totally unacceptable!",
    "The meeting starts at 10 AM tomorrow in conference room 2.",
    "Just finished my project. Feeling exhausted but satisfied."
]

print('🔮 Sentiment Predictions on New Tweets:')
print('=' * 60)
for tweet in test_tweets:
    predict_sentiment(tweet)

## 📊 Step 7: Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0f172a')
fig.suptitle('🐦 Twitter Sentiment Analysis', color='white', fontsize=16, fontweight='bold')

# Plot 1: Sentiment Distribution
colors = ['#22c55e', '#ef4444', '#6366f1']
sentiment_counts = df['sentiment'].value_counts()
axes[0].pie(sentiment_counts.values, labels=sentiment_counts.index,
            colors=colors, autopct='%1.1f%%', startangle=90,
            textprops={'color': 'white', 'fontsize': 12})
axes[0].set_title('📊 Sentiment Distribution', color='white', fontsize=13, fontweight='bold')

# Plot 2: VADER Score Distribution
pos_scores = df[df['sentiment']=='positive']['vader_score']
neg_scores = df[df['sentiment']=='negative']['vader_score']
neu_scores = df[df['sentiment']=='neutral']['vader_score']
axes[1].hist(pos_scores, bins=8, alpha=0.7, color='#22c55e', label='Positive')
axes[1].hist(neg_scores, bins=8, alpha=0.7, color='#ef4444', label='Negative')
axes[1].hist(neu_scores, bins=8, alpha=0.7, color='#6366f1', label='Neutral')
axes[1].set_title('📈 VADER Score Distribution', color='white', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Compound Score')
axes[1].set_ylabel('Count')
axes[1].legend(facecolor='#1e293b', labelcolor='white')
axes[1].axvline(x=0.05, color='white', linestyle='--', alpha=0.5)
axes[1].axvline(x=-0.05, color='white', linestyle='--', alpha=0.5)

# Plot 3: Model Accuracy Comparison
models = ['VADER', 'Naive Bayes']
accuracies = [vader_acc * 100, nb_acc * 100]
bars = axes[2].bar(models, accuracies, color=['#8b5cf6', '#f59e0b'], width=0.4)
axes[2].set_title('🏆 Model Accuracy Comparison', color='white', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_ylim(0, 110)
for bar, acc in zip(bars, accuracies):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{acc:.1f}%', ha='center', color='white', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('sentiment_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print('✅ Visualizations saved!')

## ✅ Step 8: Summary

In [ ]:
print('=' * 60)
print('🐦 TWITTER SENTIMENT ANALYSIS - PROJECT SUMMARY')
print('=' * 60)
print(f'📊 Total Tweets Analyzed       : {len(df)}')
print(f'😊 Positive Tweets             : {len(df[df["sentiment"]=="positive"])}')
print(f'😠 Negative Tweets             : {len(df[df["sentiment"]=="negative"])}')
print(f'😐 Neutral Tweets              : {len(df[df["sentiment"]=="neutral"])}')
print(f'\n🤖 VADER Accuracy              : {vader_acc:.2%}')
print(f'🧠 Naive Bayes Accuracy        : {nb_acc:.2%}')
print('\n🔧 Techniques Used:')
print('   ✅ Text Preprocessing (NLTK)')
print('   ✅ VADER Sentiment Analysis')
print('   ✅ TF-IDF Vectorization')
print('   ✅ Naive Bayes Classification')
print('   ✅ Data Visualization (Matplotlib)')
print('\n📦 Libraries: pandas, numpy, nltk, scikit-learn, matplotlib')
print('=' * 60)
print('✨ Project Complete!')